# Âge cérébral sur IXI : effet du changement d'hôpital (ClinicaDL 2.0)

**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4**.

Organisation :
- les scripts sont sur Google Drive, dans `MyDrive/test-clinicadl/scripts/` (`ixi_to_bids.py`, `train_ixi_site_shift.py`) ;
- les données sont téléchargées dans `/content` (rapide, mais effacé à la fin de la session) ;
- les résultats sont enregistrés sur Drive (`MyDrive/test-clinicadl/outputs/`), pour ne rien perdre si la session coupe.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi
!python --version

## 2. Connecter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Préparer le dossier de travail

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/test-clinicadl'
WORK = '/content/test-clinicadl'

os.makedirs(f'{WORK}/scripts', exist_ok=True)
os.makedirs(f'{WORK}/data/IXI/raw', exist_ok=True)
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)

# scripts : copie depuis Drive
!cp {DRIVE}/scripts/*.py {WORK}/scripts/
# outputs : lien vers Drive (les résultats survivent à la session)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(f'{DRIVE}/outputs', f'{WORK}/outputs')

!ls -la {WORK} {WORK}/scripts

## 4. Installer les dépendances

In [ ]:
!pip install -q clinicadl monai torchio xlrd matplotlib
!python -c "import clinicadl, torch; print('ClinicaDL', clinicadl.__version__, '| CUDA', torch.cuda.is_available())"

## 5. Télécharger IXI (≈ 4,5 Go, quelques minutes sur Colab)

In [ ]:
%cd /content/test-clinicadl/data/IXI
!curl -L -s -o IXI-T1.tar http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI/IXI-T1.tar
!curl -L -s -o IXI.xls http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI/IXI.xls
!tar -xf IXI-T1.tar -C raw && rm IXI-T1.tar
!ls raw | wc -l

## 6. Conversion en BIDS + fichier des âges

In [ ]:
%cd /content/test-clinicadl
!python scripts/ixi_to_bids.py

## 7. Contrôle du prétraitement (une image par hôpital)

In [ ]:
!python scripts/train_ixi_site_shift.py --check
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/test-clinicadl/outputs/site_shift_test-IOP/check_*.png')):
    display(Image(p))

## 8. Entraînement : Guys + HH → test sur IOP

Commencer par `--epochs 2` pour vérifier, puis lancer le vrai entraînement.

In [ ]:
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 2 --batch-size 8

In [ ]:
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 50 --batch-size 8

## 9. Résultats

Les métriques des tests `in-site` et `out-of-site` sont dans le dossier MAPS sur Drive.

In [ ]:
!find /content/test-clinicadl/outputs/site_shift_test-IOP/maps -name '*.tsv' | head -50